# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities (record sets, fields, columns) are referenced by their `@id` fields, as required by the Croissant standard.

Let's print a summary of available record sets, each with its `@id`, fields, and columns.

In [ ]:
# List all record sets and their fields, columns

print('Available record sets:')
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- RecordSet Name: {rs.name} @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields and Columns:")
    for field in rs.fields:
        col_ids = [col.id for col in (field.columns or [])]
        print(f"    - Field: {field.name} @id: {field.id}, Columns: {col_ids}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Reference all data structures and entities by their `@id`.

Here, we select all record sets and extract them to separate DataFrames, accessible by their `@id`. We'll show the columns and preview the first few rows of the main record set.

In [ ]:
# Extract all record sets to DataFrames using their '@id'
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(recs)

# Select main record set for demonstration (usually the first)
main_record_set_id = record_set_ids[0]
print(f"Main Record Set @id: {main_record_set_id}")
print('Columns:', dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

We'll select a numeric field (referenced by `@id`) and filter, normalize, and group the data. Please change the field IDs if needed, referencing the outcome of Step 2 above.

In [ ]:
# You can select from the output above; here we exemplify usage:

# Set variables for field IDs
# Example (replace with actual IDs if different):
record_set_id = main_record_set_id

# Example: choose the first numeric field (replace with correct field id from overview)
df = dataframes[record_set_id]
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization
import numpy as np
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"Field {numeric_field} is not numeric and cannot be normalized.")

# Example group field (replace with a categorical field @id)
group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
group_field = group_candidates[0] if group_candidates else None

if group_field and pd.api.types.is_object_dtype(df[group_field]):
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use matplotlib for demonstration. Adjust field `@id` references as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of the selected numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# Example: Boxplot by group field if it exists
if group_field and group_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook has shown how to access and analyze a FAIR dataset described using the Croissant schema and the `mlcroissant` library. We referenced all data entities (record sets, fields, columns) using their `@id`s for reproducibility. 

Key findings include exploration of numeric fields, basic filtering/normalization, and a look at group-wise patterns. For advanced tasks, refer to the dataset metadata and Croissant specification for full field semantics.